# Exploratory Data Analysis

In [ ]:
import os
from pathlib import Path
from typing import Dict, List, Union

import altair as alt
import boto3
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown

In [ ]:
_ = alt.data_transformers.disable_max_rows()
_ = alt.renderers.set_embed_options(actions=False)

In [ ]:
PROJ_ROOT = Path.cwd().parent

In [ ]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

In [ ]:
import cc_churn.viz_altair as vzu
import r2.io_utils as r2io
from cc_churn.eda import get_grouped_churn_rate
from utils.df_utils import show_df

## About

In this notebook, we will explore the combined train and validation data for patterns that will inform ML model validation that is the next step in the ML model development process.

We will first look at the class imbalance in the dependent variable (customer churn). This will determine our approach to reliably scoring the predictions of a ML model for this business use-case.

We will also look to identify features that are strong predictors of customer churn. This will determine which features we should prioritize for use in model training, how we should pre-process the features and which family of models are likely to be better choices to predict the outcome in this dataset.

### Outputs

Charts will be saved as `.html` files in `reports/figures`.

## User Inputs

In [ ]:
# R2 data bucket details
# # name of train data key (file) in private R2 bucket
r2_key_train = "train_data.parquet.gzip"
# # name of validation data key (file) in private R2 bucket
r2_key_val = "validation_data.parquet.gzip"

# features
numerical_features = [
    "customer_age",
    "months_on_book",
    "dependent_count",
    "months_inactive_12_mon",
    "contacts_count_12_mon",
    "credit_limit",
    "total_revolv_bal",
    "avg_open_to_buy",
    "total_amt_chng_q4_q1",
    "total_trans_amt",
    "total_trans_ct",
    "total_ct_chng_q4_q1",
    "avg_utilization_ratio",
]

# datatypes for categorical and ordinal columns
dtypes_ordinals = {
    "gender": "string[pyarrow]",
    "income_category": "string[pyarrow]",
    "education_level": "string[pyarrow]",
}
dtypes_categoricals = {
    "marital_status": "string[pyarrow]",
    "card_category": "string[pyarrow]",
}

label = "is_churned"

threshold_correlation = 0.55

In [ ]:
reports_dir = PROJ_ROOT / "reports"
figures_dir = reports_dir / "figures"

account_id = os.getenv("ACCOUNT_ID")
access_key_id = os.getenv("ACCESS_KEY_ID_USER2")
secret_access_key = os.getenv("SECRET_ACCESS_KEY_USER2")
bucket_name = os.getenv("BUCKET_NAME")

s3_client = boto3.client(
    "s3",
    endpoint_url=f"https://{account_id}.r2.cloudflarestorage.com",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    region_name="auto",
)

## Get Data

In [ ]:
%%time
df = (
    pd.concat(
        [
            r2io.pandas_read_parquet_r2(s3_client, bucket_name, key)
            for key in [r2_key_train, r2_key_val]
        ]
    )
    .assign(
        is_churned=lambda df: df['is_churned'].map(
            {1: 'Churned', 0: 'Did not Churn'}
        )
    )
    .rename(columns={"is_churned": 'outcome'})
    .astype(dtypes_ordinals)
    .astype(dtypes_categoricals)
)
print(f"Loaded {len(df):,} rows of data")
dfd = show_df(df)
with pd.option_context('display.max_columns', None):
    display(df.head(1))

# Exploratory Data Analysis

We will explore the combined training and validation data based on the type of features in three steps

1. we will explore the class imbalance in the outcome (dependent variable, churn versus no churn)
2. we will visualize numerical features using grouped histograms, in order to investigate their distrubtions among churned and non-churned (i.e. *safe*) customers
3. we will then explore categorical and ordinal feature counts for both types of customers, using bar charts

### Class Imbalance

Show the class imbalance

In [ ]:
display(df["outcome"].value_counts(normalize=True).to_frame())

**Observations**

1. The class labels show a medium class imbalance. Standard ML classifiers typically aim to maximize overall accuracy. Unfortunately, in an imbalanced context, this often results in a model that predicts the majority class for every customer. These predictions would have high accuracy, but it would have zero recall for the churned customers, making it useless for our business goal. This happens since the default decision threshold of 0.5 fails since it assumes that the costs of a False Positive (predicting a loyal customer will churn) and a False Negative (failing to identify a customer who actually churns) are equal. For customer churn, the cost of False Negatives, or the cost of losing a customer's lifetime value and the cost of acquiring a new one, is high. Also, the cost of False Positives, the cost of a retention offer or a phone call to a customer who wasn't planning to leave, is lower. So, the default decision threshold of 0.5 is of no use to use here. We will need to optimize this threshold.

   In order to solve this problem, we need to move beyond simple accuracy and toward [cost-sensitive learning](https://machinelearningmastery.com/cost-sensitive-learning-for-imbalanced-classification/). In `scikit-learn` in Python, one approach is to use nested cross-validation and `TunedThresholdClassifierCV()`. The inner cross-validation loop optimizes the decision threshold. This allows the model to become more sensitive to the (minority) churn class. By nesting this process within an outer cross-validation loop, we ensure that the performance metrics we report are not overfitted to the specific threshold found. The outer loop validates how well the entire 'tuning process' generalizes to unseen data.

   Using `.predict_proba()` allows us to rank customers by churn risk, for the client. But, by determining an optimal decision threshold, we can reliably convert these *soft labels* into *hard labels* that we can use to reliably identify those customers who are truly at risk of canceling their credit card services at the bank. Customers with a predicted probability above this threshold are at risk of churning, while those below are not at risk. This approach will ensure the client's targeting resources are allocated efficiently on those at risk of churning only.
2. For `scikit-learn` models, we can use `class_weight="balanced"`. For `XFBClassifier()`, we can use `scale_pos_weight` to perform cost-sensitive learning.

### Churn Analysis for Numerical Features

In [ ]:
legend_params_histogram = dict(
    orient="bottom",
    columns=2,
    labelAlign="left",
    titleAnchor="start",
    labelFontSize=15,
)

#### Customer Age

In [ ]:
chart = vzu.plot_grouped_overlapping_altair_histogram(
    df,
    num_bins=30,
    xvar="customer_age:Q",
    xtitle="Customer Age",
    ytitle=None,
    color_by_col="outcome:N",
    legend_title=None,
    scale_params=dict(
        domain=["Did not Churn", "Churned"], range=["lightgrey", "darkred"]
    ),
    ptitle=alt.TitleParams(
        text="Churned and Safe Customer Ages are Normally Distributed",
        anchor="start",
        align="left",
        dx=40,
        fontSize=18,
    ),
    y_scale="symlog",
    tooltip=["outcome", "count()"],
    fig_size=dict(width=750, height=300),
    save_params=dict(fpath=figures_dir / "fig_03_eda_age_histogram.html"),
).configure_legend(**legend_params_histogram)
chart

**Observations**

1. Churn is fairly normally distributed across age groups, suggesting age is a primary driver of churn.

#### Customer Tenure

In [ ]:
chart = vzu.plot_grouped_overlapping_altair_histogram(
    df,
    num_bins=30,
    xvar="months_on_book:Q",
    xtitle="Months on Book",
    ytitle=None,
    color_by_col="outcome:N",
    legend_title=None,
    scale_params=dict(
        domain=["Did not Churn", "Churned"], range=["lightgrey", "darkred"]
    ),
    ptitle=alt.TitleParams(
        text="Churned and Safe Customer Tenures are Normally Distributed",
        anchor="start",
        align="left",
        dx=40,
        fontSize=18,
    ),
    y_scale="symlog",
    tooltip=["outcome", "count()"],
    fig_size=dict(width=750, height=300),
    save_params=dict(fpath=figures_dir / "fig_04_eda_tenure_histogram.html"),
).configure_legend(**legend_params_histogram)
chart

**Observations**

1. Retention is also normally distributed across customer tenures with the bank's credit card division (in number of months). A strong peak appears around the 36-month mark. More customers canceled their credit card after approximately 36 months than any other tenure duration.

#### Credit Limit (Dollars)

In [ ]:
chart = vzu.plot_grouped_overlapping_altair_histogram(
    df,
    num_bins=30,
    xvar="credit_limit:Q",
    xtitle="Credit Limit ($)",
    ytitle=None,
    color_by_col="outcome:N",
    legend_title=None,
    scale_params=dict(
        domain=["Did not Churn", "Churned"], range=["lightgrey", "darkred"]
    ),
    ptitle=alt.TitleParams(
        text=(
            "Credit Limit has Right Skew with Increase at Approximately $36,000"
        ),
        anchor="start",
        align="left",
        dx=40,
        fontSize=18,
    ),
    y_scale="symlog",
    tooltip=["outcome", "count()"],
    fig_size=dict(width=750, height=300),
    save_params=dict(
        fpath=figures_dir / "fig_05_eda_credit_limit_histogram.html"
    ),
).configure_legend(**legend_params_histogram)
chart

**Observations**

1. Churners are slightly more concentrated in the lower credit limit brackets or at approximately 36,000 dollars. Both types of customers follow the same right-skewed distribution. This looks like a weak predictor of customer churn.

#### Credit Card Debt (Dollars)

In [ ]:
chart = vzu.plot_grouped_overlapping_altair_histogram(
    df,
    num_bins=30,
    xvar="total_revolv_bal:Q",
    xtitle="Credit Limit ($)",
    ytitle=None,
    color_by_col="outcome:N",
    legend_title=None,
    scale_params=dict(
        domain=["Did not Churn", "Churned"], range=["lightgrey", "darkred"]
    ),
    ptitle=alt.TitleParams(
        text=(
            "Churners have Either a Low ($0-600) or High (~$2,600) in Credit "
            "Card Debt"
        ),
        anchor="start",
        align="left",
        dx=40,
        fontSize=18,
    ),
    y_scale="symlog",
    tooltip=["outcome", "count()"],
    fig_size=dict(width=750, height=300),
    save_params=dict(
        fpath=figures_dir / "fig_06_eda_total_revolv_bal_histogram.html"
    ),
).configure_legend(**legend_params_histogram)
chart

**Observations**

1. Churned customers often have a total revolving balance in the $0-600 range, indicating a lack of card usage, or at  approximately 2,600 dollars. In between these two extremes, the amount of debt shows a uniform distribution for both types of customers. The amount of credit card debt also looks like a predictor of churn.

#### Amount of Credit Available to Use (Dollars)

In [ ]:
chart = vzu.plot_grouped_overlapping_altair_histogram(
    df,
    num_bins=30,
    xvar="avg_open_to_buy:Q",
    xtitle="Amount Left on Credit Card to Use ($) Over Last 12 Months",
    ytitle=None,
    color_by_col="outcome:N",
    legend_title=None,
    scale_params=dict(
        domain=["Did not Churn", "Churned"], range=["lightgrey", "darkred"]
    ),
    ptitle=alt.TitleParams(
        text=(
            "Churners are More Prevalent in Lower open-to-buy Ranges or at "
            "~$36,000"
        ),
        anchor="start",
        align="left",
        dx=40,
        fontSize=18,
    ),
    y_scale="symlog",
    tooltip=["outcome", "count()"],
    fig_size=dict(width=750, height=300),
    save_params=dict(
        fpath=figures_dir / "fig_07_eda_total_avg_open_to_buy_histogram.html"
    ),
).configure_legend(**legend_params_histogram)
chart

**Observations**

1. Similar to credit limit, churners are more prevalent in the lower 'open to buy' ranges. There is a strong increase at approximately 36,000 dollars and most customers in this bracket are churners, which suggests a lack of credit card usage.
2. Based on the distribution, this feature might be correlated to credit limit. This should be explicitly checked later.

#### Recent Transaction Amount Ratio

In [ ]:
chart = vzu.plot_grouped_overlapping_altair_histogram(
    df,
    num_bins=30,
    xvar="total_amt_chng_q4_q1:Q",
    xtitle="Change in Credit Card Transaction Amount (Q4 / Q1)",
    ytitle=None,
    color_by_col="outcome:N",
    legend_title=None,
    scale_params=dict(
        domain=["Did not Churn", "Churned"], range=["lightgrey", "darkred"]
    ),
    ptitle=alt.TitleParams(
        text="Churners Have a Lower Q4 / Q1 Transaction Amount Ratio",
        anchor="start",
        align="left",
        dx=40,
        fontSize=18,
    ),
    y_scale="symlog",
    tooltip=["outcome", "count()"],
    fig_size=dict(width=750, height=300),
    save_params=dict(
        fpath=figures_dir / "fig_08_eda_total_amt_chng_histogram.html"
    ),
).configure_legend(**legend_params_histogram)
chart

**Observations**

1. Churners tend to have a lower ratio, with a maximum value of ~1.6, compared to ~3.4 for non-churners. Churners are more strongly concentrated in the 0-0.6 range than non-churners. This indicates a decrease in spending amount recently for customers who canceled their credit card services.
2. Both types of customers have a left-skewed distrubtion.
3. Based on observation 1., this feature looks like a predictor of churn.

#### Credit Card Transactions (Dollars)

In [ ]:
chart = vzu.plot_grouped_overlapping_altair_histogram(
    df,
    num_bins=30,
    xvar="total_trans_amt:Q",
    xtitle="Total Transaction Amount (Last 12 months)",
    ytitle=None,
    color_by_col="outcome:N",
    legend_title=None,
    scale_params=dict(
        domain=["Did not Churn", "Churned"], range=["lightgrey", "darkred"]
    ),
    ptitle=alt.TitleParams(
        text=(
            "Churners Have Less Than ~$11,000 in Credit Card Transactions Over "
            "Last 12 Months"
        ),
        anchor="start",
        align="left",
        dx=40,
        fontSize=18,
    ),
    y_scale="symlog",
    tooltip=["outcome", "count()"],
    fig_size=dict(width=750, height=300),
    save_params=dict(
        fpath=figures_dir / "fig_09_eda_total_trans_amt_histogram.html"
    ),
).configure_legend(**legend_params_histogram)
chart

**Observations**

1. Churners are heavily concentrated in the lower total transaction amount range (2,000-3,000 dollars) with a maximum of ~11,000 dollars.
2. On the low end, non-churned customers have an average of ~5,000 dollars. Notably, they show a bi-modal distribution not seen in churners. The second peak has an average of approximately 15,000 dollars.
3. This feature appears to be a predictor of churn.

#### Number of Credit Card Transactions in Last 12 Months

In [ ]:
chart = vzu.plot_grouped_overlapping_altair_histogram(
    df,
    num_bins=30,
    xvar="total_trans_ct:Q",
    xtitle="Total Number of Credit Card Transactions (Last 12 months)",
    ytitle=None,
    color_by_col="outcome:N",
    legend_title=None,
    scale_params=dict(
        domain=["Did not Churn", "Churned"], range=["lightgrey", "darkred"]
    ),
    ptitle=alt.TitleParams(
        text=(
            "Churners Performed ~50% Fewer Credit Card Transactions in "
            "Last 12 Months Than Others"
        ),
        anchor="start",
        align="left",
        dx=40,
        fontSize=17,
    ),
    y_scale="symlog",
    tooltip=["outcome", "count()"],
    fig_size=dict(width=750, height=300),
    save_params=dict(
        fpath=figures_dir / "fig_10_eda_total_trans_cnt_histogram.html"
    ),
).configure_legend(**legend_params_histogram)
chart

**Observations**

1. Churners show significantly lower transaction counts, usually peaking below 50 transactions. This is approximately 30-50% of the average credit card activity of other (safe) customers.
2. Similar to credit card transaction amount (`total_trans_amt`), this feature has a bi- or [tri-modal](https://www.researchgate.net/figure/Figure-1-Tri-modal-Histogram_fig1_221551908) pattern with a distinct second peak at approximately 80 transactions and a third peak at approximately 120 transactions. These two peaks are not present for churned customers.
3. This feature looks like a predictor of churn.

#### Change in Recent Credit Card Transaction Count (Ratio of Q4 / Q1)

In [ ]:
chart = vzu.plot_grouped_overlapping_altair_histogram(
    df,
    num_bins=30,
    xvar="total_ct_chng_q4_q1:Q",
    xtitle="Change in Total Credit Card Transaction Count (Q4 / Q1)",
    ytitle=None,
    color_by_col="outcome:N",
    legend_title=None,
    scale_params=dict(
        domain=["Did not Churn", "Churned"], range=["lightgrey", "darkred"]
    ),
    ptitle=alt.TitleParams(
        text=(
            "Churners Have Less Than ~$11,000 in Credit Card Transactions Over "
            "Last 12 Months"
        ),
        anchor="start",
        align="left",
        dx=40,
        fontSize=17,
    ),
    y_scale="symlog",
    tooltip=["outcome", "count()"],
    fig_size=dict(width=750, height=300),
    save_params=dict(
        fpath=figures_dir / "fig_11_eda_total_ct_chng_q4_q1_histogram.html"
    ),
).configure_legend(**legend_params_histogram)
chart

**Observations**

1. A lower count change ratio (below 0.6).
2. The distribution of this feature is right-skewed and is similar to that of `total_amt_chng_q4_q1`.
3. As with `total_amt_chng_q4_q1`, this feature appers to be an indicator of customer churn.

#### Ratio of Credit Usage to Available Credit Limit in Last 12 Months

In [ ]:
chart = vzu.plot_grouped_overlapping_altair_histogram(
    df,
    num_bins=30,
    xvar="avg_utilization_ratio:Q",
    xtitle=(
        "Average Ratio of Credit Usage Relative to Total Credit Limit "
        "(last 12 months)"
    ),
    ytitle=None,
    color_by_col="outcome:N",
    legend_title=None,
    scale_params=dict(
        domain=["Did not Churn", "Churned"], range=["lightgrey", "darkred"]
    ),
    ptitle=alt.TitleParams(
        text=(
            "Churners Have Have an Average Utilization Ratio of 0.0 Over "
            "Last 12 Months"
        ),
        anchor="start",
        align="left",
        dx=40,
        fontSize=17,
    ),
    y_scale="symlog",
    tooltip=["outcome", "count()"],
    fig_size=dict(width=750, height=300),
    save_params=dict(
        fpath=figures_dir / "fig_12_eda_avg_util_ratio_histogram.html"
    ),
).configure_legend(**legend_params_histogram)
chart

**Observations**

1. Churners predominantly have a near-zero utilization ratio, highlighting a lack of engagement with the bank's credit card offering. This is in line with the lack-of-engagement characteristics of churned customers seen from the other features.
2. Excluding a ratio of 0, the distribution of the non-churned customers is right-skewed, while that of churners is approximately uniform.
3. Based on these observations, the `avg_utilization_ratio` feature looks like a predictor of churn.

#### Number of Products (Accounts, Credit lines, etc.) held by Customer with Bank

In [ ]:
chart = vzu.plot_altair_bar_chart(
    get_grouped_churn_rate(df, "num_products"),
    xvar="frac_churned:Q",
    xvar2="total:Q",
    yvar="num_products:N",
    xtitle="Fraction of Churned Customers (%)",
    xtitle2="Number of Customers",
    ytitle=None,
    y_sort="-x",
    tooltip=[
        "num_products",
        "Churned",
        "Did not Churn",
        alt.Tooltip(
            "frac_churned:Q", title="Fraction Churned (%)", format=",.2f"
        ),
    ],
    tooltip2=[
        "num_products",
        alt.Tooltip("total:Q", title="Number of Customers", format=","),
        alt.Tooltip(
            "frac_total:Q", title="Fraction of Customers (%)", format=",.2f"
        ),
    ],
    ptitle=alt.TitleParams(
        text=(
            "Generally, Churn Rate Decreases as Customers Sign-Up to More Bank "
            "Products"
        ),
        anchor="start",
        align="left",
        dx=15,
        fontSize=18,
    ),
    x_scale="linear",
    fig_size=dict(width=407, height=125),
    save_params=dict(fpath=figures_dir / "fig_13_eda_num_products_bar.html"),
)
chart

**Observations**

1. Customer with 3 bank products have a churn rate of ~18%. Adding 1-3 more products results in the churn rate dropping by five percentage points. Reducing the number of products to 1-2 results in churn increasing by nearly 10 percentage points to 27-28%. In summary, customers with fewer products (1-2) show a higher proportion of churn compared to those with 4-6 products.
2. This feature appears to be a predictor or churn.

#### Number of Months Customer was Inactive in the Last 12 Months

In [ ]:
chart = vzu.plot_altair_bar_chart(
    get_grouped_churn_rate(df, "months_inactive_12_mon"),
    xvar="frac_churned:Q",
    xvar2="total:Q",
    yvar="months_inactive_12_mon:N",
    xtitle="Fraction of Churned Customers (%)",
    xtitle2="Number of Customers",
    ytitle=None,
    y_sort="-x",
    tooltip=[
        "months_inactive_12_mon",
        "Churned",
        "Did not Churn",
        alt.Tooltip(
            "frac_churned:Q", title="Fraction Churned (%)", format=",.2f"
        ),
    ],
    tooltip2=[
        "months_inactive_12_mon",
        alt.Tooltip("total:Q", title="Number of Customers", format=","),
        alt.Tooltip(
            "frac_total:Q", title="Fraction of Customers (%)", format=",.2f"
        ),
    ],
    ptitle=alt.TitleParams(
        text=(
            "Churn Notably Increases for Customers who are Inactive for 3+ "
            "Months"
        ),
        anchor="start",
        align="left",
        dx=15,
        fontSize=18,
    ),
    x_scale="linear",
    fig_size=dict(width=407, height=125),
    save_params=dict(
        fpath=figures_dir / "fig_14_eda_months_inactive_12_mon_bar.html"
    ),
)
chart

**Observations**

1. Churn significantly increases when customers are inactive for 3 or more months.
2. This feature looks like a predictor of churn.

#### Number of Times Customer Contacted (or Communicated With) Bank in Last 12 Months

In [ ]:
chart = vzu.plot_altair_bar_chart(
    get_grouped_churn_rate(df, "contacts_count_12_mon"),
    xvar="frac_churned:Q",
    xvar2="total:Q",
    yvar="contacts_count_12_mon:N",
    xtitle="Fraction of Churned Customers (%)",
    xtitle2="Number of Customers",
    ytitle=None,
    y_sort="-x",
    tooltip=[
        "contacts_count_12_mon",
        "Churned",
        "Did not Churn",
        alt.Tooltip(
            "frac_churned:Q", title="Fraction Churned (%)", format=",.2f"
        ),
    ],
    tooltip2=[
        "contacts_count_12_mon",
        alt.Tooltip("total:Q", title="Number of Customers", format=","),
        alt.Tooltip(
            "frac_total:Q", title="Fraction of Customers (%)", format=",.2f"
        ),
    ],
    ptitle=alt.TitleParams(
        text=(
            "Generally, Churn Rate Decreases as Customers Sign-Up to More Bank "
            "Products"
        ),
        anchor="start",
        align="left",
        dx=15,
        fontSize=18,
    ),
    x_scale="linear",
    fig_size=dict(width=407, height=125),
    save_params=dict(
        fpath=figures_dir / "fig_15_eda_contacts_count_12_mon_bar.html"
    ),
)
chart

**Observations**

1. As the number of bank contacts (communications) increased from one to four during the last 12 months, customer churn rate increased from ~7.5% to ~22%. This likely indicates issues that were not resolved during a customer's communication with the bank.
2. This feature appears to be a predictor of churn.

#### Feature Collinearity

Collinearity occurs when two or more predictor variables are highly correlated. This means they provide redundant information. In ML models, this can lead to unstable coefficient estimates (in linear models) or diluted feature importance (in tree-based models). When features show near-perfect correlation, keeping both makes it difficult to determine which specific variable is actually driving the prediction.

For a bank, a *black box* prediction is only partially useful. Explainability allows the client to
1. Understanding whether a customer is prone to churn due to low engagement (`total_trans_ct`) versus declining spend (`total_amt_chng_q4_q1`). This difference is important since it allows for the client to develop personalized retention offers.
2. In financial services, the ability to explain why a certain action (or lack thereof) was taken regarding a customer account is often a legal or internal audit requirement.

By excluding redundant, highly correlated features, we can make it easier for the model to extract clear, actionable insights using tools like SHAP or feature importance rankings.

In [ ]:
Markdown(
    f"""For these reasons, we will drop one feature from each pair of
    correlated numerical features. The threshold for correlation will be
    assumed to be {threshold_correlation:.2f}.
    """
)

Below is a feature correlation heatmap (reds indicate stronger positive correlation andblues indicate stronger negative correlation)

In [ ]:
%%time
df_corr = df[numerical_features].corr(method="pearson")
corr_matrix = df_corr.reset_index().melt(id_vars="index")
corr_matrix.columns = ["var1", "var2", "correlation"]
col_order = {col: i for i, col in enumerate(numerical_features)}
corr_matrix = corr_matrix[
    corr_matrix.apply(
        lambda x: col_order[x["var1"]] >= col_order[x["var2"]], axis=1
    )
]

In [ ]:
chart = vzu.plot_altair_heatmap(
    corr_matrix,
    xvar="var1:N",
    yvar="var2:N",
    textvar="correlation:Q",
    xsort=numerical_features,
    ysort=numerical_features,
    color_by_col="correlation:Q",
    border_attrs=dict(stroke="white", strokeWidth=2),
    scale_params=dict(scheme="redblue", domain=[1, -1]),
    text_fontsize=14,
    text_alt_condition="abs(datum.correlation) > 0.5",
    tooltip=[alt.Tooltip("correlation:Q", title='Correlation')],
    ptitle=alt.TitleParams(
        text="Four Feature Pairs are Highly Correlated (>0.55, in Darker Red)",
        anchor="start",
        align="left",
        dx=185,
        fontSize=18,
    ),
    fig_size=dict(width=700, height=375),
    save_params=dict(fpath=figures_dir / "fig_16_eda_correlation.html"),
)
chart

Below are combinations of features that are correlated above the assumed threshold of 0.55

In [ ]:
%%time
high_corr_pairs = []
for i in range(len(df_corr.columns)):
    for j in range(i + 1, len(df_corr.columns)):
        col1 = df_corr.columns[i]
        col2 = df_corr.columns[j]
        corr_value = df_corr.loc[col1, col2]
        if abs(corr_value) > threshold_correlation:
            high_corr_pairs.append(
                {
                    "feature_1": col1,
                    "feature_2": col2,
                    "correlation": corr_value,
                }
            )
df_correlated_features = pd.DataFrame.from_records(
    high_corr_pairs
).sort_values(by=["correlation"], ascending=False, ignore_index=True)
display(
    df_correlated_features.style.set_properties(
        subset=["correlation"], **{"background-color": "yellow"}
    )
)

#### Bivariate Analysis - Transaction Count versus Dollar Value

In [ ]:
chart = vzu.plot_altair_scatter_chart(
    df,
    xvar="total_trans_ct:Q",
    yvar="total_trans_amt:Q",
    xtitle="Total Transaction Count",
    ytitle="Total Transaction Dollar Value",
    color_by_col="outcome:N",
    ptitle=alt.TitleParams(
        text=(
            "Churners Perform Fewer Transactions and for Lower Dollar Value"
        ),
        anchor="start",
        align="left",
        dx=15,
        fontSize=18,
    ),
    xscale="linear",
    yscale="linear",
    scale_params=dict(
        domain=["Churned", "Did not Churn"], range=["darkred", "lightgrey"]
    ),
    fig_size=dict(width=500, height=400),
    save_params=dict(
        fpath=figures_dir / "fig_17_eda_total_trans_ct_vs_amt_scatter.html"
    ),
)
chart

**Observations**

1. Churners are tightly clustered in the bottom-left corner (low volume, low value), suggesting they stop using the card before leaving.

#### Bivariate Analysis - Ratio of Recent Change in Transaction Count versus Dollar Value

In [ ]:
chart = vzu.plot_altair_scatter_chart(
    df,
    xvar="total_ct_chng_q4_q1:Q",
    yvar="total_amt_chng_q4_q1:Q",
    xtitle="Ratio of Total Transaction Count in Q4 to Q1",
    ytitle="Ratio of Total Transaction Amount in Q4 to Q1",
    color_by_col="outcome:N",
    ptitle=alt.TitleParams(
        text=(
            "Over the Course of the Year, Churners Performed Fewer "
            "Credit Card Transactions and for a Lower Dollar Value"
        ),
        anchor="start",
        align="left",
        dx=15,
        fontSize=18,
    ),
    xscale="linear",
    yscale="linear",
    scale_params=dict(
        domain=["Churned", "Did not Churn"], range=["darkred", "lightgrey"]
    ),
    fig_size=dict(width=500, height=400),
    save_params=dict(
        fpath=figures_dir
        / "fig_18_eda_total_ct_chng_q4_q1_vs_amt_scatter.html"
    ),
)
chart

**Observations**

1. A clear cluster of churners exists where count ratio is below approximately 0.6 and amount change ratio is below approximately 1.0. This confirms our earlier observations of a simultaneous drop in credit card transaction frequency and spend.

#### Bivariate Analysis - Revolving Balance versus Credit Utilization Ratio

In [ ]:
chart = vzu.plot_altair_scatter_chart(
    df,
    xvar="total_revolv_bal:Q",
    yvar="avg_utilization_ratio:Q",
    xtitle="Carryover Credit Card Balance",
    ytitle="Ratio of Carryover Balance to Credit Limit",
    color_by_col="outcome:N",
    ptitle=alt.TitleParams(
        text="Churners Spend Less of Available Credit than Other Customers",
        anchor="start",
        align="left",
        dx=15,
        fontSize=18,
    ),
    xscale="linear",
    yscale="linear",
    scale_params=dict(
        domain=["Churned", "Did not Churn"], range=["darkred", "lightgrey"]
    ),
    fig_size=dict(width=500, height=400),
    save_params=dict(
        fpath=figures_dir / "fig_19_eda_total_revolv_bal_vs_avg_util_ratio_"
        "scatter.html"
    ),
)
chart

**Observations**

1. Churners are concentrated at the (0,0) origin point, which corresponds to a low number of credit card transactions and dollar value spent. Non-churned customers are not found in this section of the plot. This confirms these two features are useful predictors of churn.

#### Bivariate Analysis - Credit Limit versus Utilization Ratio

In [ ]:
chart = vzu.plot_altair_scatter_chart(
    df,
    xvar="credit_limit:Q",
    yvar="avg_utilization_ratio:Q",
    xtitle="Credit Limit",
    ytitle="Ratio of Carryover Balance to Credit Limit",
    color_by_col="outcome:N",
    ptitle=alt.TitleParams(
        text="Churners Utilization is Generally Below Other Customers",
        anchor="start",
        align="left",
        dx=15,
        fontSize=18,
    ),
    xscale="linear",
    yscale="linear",
    scale_params=dict(
        domain=["Churned", "Did not Churn"], range=["darkred", "lightgrey"]
    ),
    fig_size=dict(width=500, height=400),
    save_params=dict(
        fpath=(
            figures_dir / "fig_20_eda_credit_limit_vs_avg_util_ratio_"
            "scatter.html"
        )
    ),
)
chart

**Observations**

1. Churners mostly occupy the low-utilization band regardless of credit card limit. Churners are more frequent in low-limit accounts. Again, there is a clear separation between churners and non-churners, which reaffirms the usefulness of these two features to predict churn.

#### Bivariate Analysis - Customer Age versus Months on Book at the Bank

In [ ]:
chart = vzu.plot_altair_scatter_chart(
    df,
    xvar="customer_age:Q",
    yvar="months_on_book:Q",
    xtitle="Customer Age",
    ytitle="Months on Book (Period of Relationship with Bank)",
    color_by_col="outcome:N",
    ptitle=alt.TitleParams(
        text=(
            "No Separation Between Churners and Others, in terms of Age and "
            "Months on Book at the Bank"
        ),
        anchor="start",
        align="left",
        dx=15,
        fontSize=18,
    ),
    xscale="linear",
    yscale="linear",
    scale_params=dict(
        domain=["Churned", "Did not Churn"], range=["darkred", "lightgrey"]
    ),
    fig_size=dict(width=500, height=400),
    save_params=dict(
        fpath=(
            figures_dir
            / ("fig_21_eda_customer_age_vs_months_on_book_scatter.html")
        )
    ),
)
chart

**Observations**

1. There is a strong linear relationship between age and months on book. This is as expected.
2. Churners are distributed across the entire range. This suggests that tenure alone isn't a simple predictor without behavioral context. Either feature on its own won't be effective at predicting churn.

### Analysis of Categorical and Ordinal Features

Below is the number of unique values in all categorical and ordinal features

In [ ]:
(
    df[list(dtypes_categoricals) + list(dtypes_ordinals)]
    .nunique()
    .reset_index()
    .rename(columns={"index": "feature_name", 0: "num_unique_values"})
)

High cardinality is not observed in any of the categorical or ordinal features. Categorical features can be one-hot encoded with minimal feature expansions. This has several benefits

1. it keeps the encoded data dense, thereby improving the speed of model training
2. it is easier to interpret
3. it avoids the [curse of dimensionality when performing one-hot encoding](https://apxml.com/courses/intro-feature-engineering/chapter-3-encoding-categorical-features/high-cardinality-features)

So, all categorical and ordinal features can be used in model development.

#### Customer Gender

In [ ]:
chart = vzu.plot_altair_bar_chart(
    get_grouped_churn_rate(df, "gender"),
    xvar="frac_churned:Q",
    xvar2="total:Q",
    yvar="gender:N",
    xtitle="Fraction of Churned Customers (%)",
    xtitle2="Number of Customers",
    ytitle=None,
    y_sort=["F", "M"],
    tooltip=[
        "gender",
        "Churned",
        "Did not Churn",
        alt.Tooltip(
            "frac_churned:Q", title="Fraction Churned (%)", format=",.2f"
        ),
    ],
    tooltip2=[
        "gender",
        alt.Tooltip("total:Q", title="Number of Customers", format=","),
        alt.Tooltip(
            "frac_total:Q", title="Fraction of Customers (%)", format=",.2f"
        ),
    ],
    ptitle=alt.TitleParams(
        text=(
            "Churn Rate and Frequencies are Similar Across Both Genders in "
            "the Customer Data"
        ),
        anchor="start",
        align="left",
        dx=15,
        fontSize=18,
    ),
    x_scale="linear",
    fig_size=dict(width=375, height=50),
    save_params=dict(fpath=figures_dir / "fig_22_eda_gender_bar.html"),
)
chart

**Observations**

1. Female customers show a slightly higher count and proportion (approximately 2.5 percentage points) of churn compared to males.
2. Gender looks like a weak predictor of churn.

#### Marital Status of Customers

In [ ]:
chart = vzu.plot_altair_bar_chart(
    get_grouped_churn_rate(df, "marital_status"),
    xvar="frac_churned:Q",
    xvar2="total:Q",
    yvar="marital_status:N",
    xtitle="Fraction of Churned Customers (%)",
    xtitle2="Number of Customers",
    ytitle=None,
    y_sort="-x",
    tooltip=[
        "marital_status",
        "Churned",
        "Did not Churn",
        alt.Tooltip(
            "frac_churned:Q", title="Fraction Churned (%)", format=",.2f"
        ),
    ],
    tooltip2=[
        "marital_status",
        alt.Tooltip("total:Q", title="Number of Customers", format=","),
        alt.Tooltip(
            "frac_total:Q", title="Fraction of Customers (%)", format=",.2f"
        ),
    ],
    ptitle=alt.TitleParams(
        text=(
            "For Known Marital Status, Churn Rate is Similar Regardless of "
            "Marital Status of Customers"
        ),
        anchor="start",
        align="left",
        dx=65,
        fontSize=18,
    ),
    x_scale="linear",
    fig_size=dict(width=407, height=125),
    save_params=dict(fpath=figures_dir / "fig_23_eda_gender_matital.html"),
)
chart

**Observations**

1. Churn is distributed across all statuses, but *Married* and *Single* customers make up the largest fraction of the bank's credit card customer base.
2. The churn rate doesn't show a significant visual variance based on marital status.
3. <font color='darkred'>**(Re-Grouping Rare Categories)**</font> The two rare categories, *Unknown* and *Divorced*, should be combined into *Other*. This would cause all sub-categories to occur with >15% frequency in the combined train+validation data, and eliminates two which each have a frequency of <~7.4%.
4. For the two most popular categories in this feature, churn rate changes from ~14.9% (*Married*) to ~16.5% (*Divorced*) which is less than two percentage points. So, this feature also appears to be a weak predictor of credit card churn.

#### Category of Customers' Credit Card

In [ ]:
chart = vzu.plot_altair_bar_chart(
    get_grouped_churn_rate(df, "card_category"),
    xvar="frac_churned:Q",
    xvar2="total:Q",
    yvar="card_category:N",
    xtitle="Fraction of Churned Customers (%)",
    xtitle2="Number of Customers",
    ytitle=None,
    y_sort="-x",
    tooltip=[
        "card_category",
        "Churned",
        "Did not Churn",
        alt.Tooltip(
            "frac_churned:Q", title="Fraction Churned (%)", format=",.2f"
        ),
    ],
    tooltip2=[
        "card_category",
        alt.Tooltip("total:Q", title="Number of Customers", format=","),
        alt.Tooltip(
            "frac_total:Q", title="Fraction of Customers (%)", format=",.2f"
        ),
    ],
    ptitle=alt.TitleParams(
        text="Churn Rate is Higher for Two Most Infrequent Card Categories",
        anchor="start",
        align="left",
        dx=65,
        fontSize=18,
    ),
    x_scale="linear",
    fig_size=dict(width=425, height=125),
    save_params=dict(fpath=figures_dir / "fig_24_eda_card_category_bar.html"),
)
chart

**Observations**

1. The majority of customers are in the *Blue* credit card category. Higher-tier cards have fewer customers. All categories of cards don't visually suggest a lower churn rate proportion at a glance.
2. <font color='darkred'>**(Handling Rare Categories)**</font> The value *Blue* accounts for approximately 93% of all customers in the combined train+validation data. This is almost entirely a single-valued feature that does not have much predictive power. So, we will not use this feature in modeling.
3. With predominantly a single unique value, this looks like a weak predictor of churn.

#### Number of Customers' Dependents

In [ ]:
chart = vzu.plot_altair_bar_chart(
    get_grouped_churn_rate(df, "dependent_count"),
    xvar="frac_churned:Q",
    xvar2="total:Q",
    yvar="dependent_count:N",
    xtitle="Fraction of Churned Customers (%)",
    xtitle2="Number of Customers",
    ytitle=None,
    y_sort="-x",
    tooltip=[
        "dependent_count",
        "Churned",
        "Did not Churn",
        alt.Tooltip(
            "frac_churned:Q", title="Fraction Churned (%)", format=",.2f"
        ),
    ],
    tooltip2=[
        "dependent_count",
        alt.Tooltip("total:Q", title="Number of Customers", format=","),
        alt.Tooltip(
            "frac_total:Q", title="Fraction of Customers (%)", format=",.2f"
        ),
    ],
    ptitle=alt.TitleParams(
        text=(
            "Churn Rate is Separated by At Most 3 Percentage Points "
            "Regardless of Number of Dependents"
        ),
        anchor="start",
        align="left",
        dx=10,
        fontSize=18,
    ),
    x_scale="linear",
    fig_size=dict(width=412, height=125),
    save_params=dict(fpath=figures_dir / "fig_25_eda_dependent_bar.html"),
)
chart

**Observations**

1. Customers with 2 or 3 dependents are the most common among the sample of credit card customers in the data we have.
2. Churn counts appear to scale proportionally with the size of the segment.
3. <font color='darkred'>**(Re-Grouping Rare Categories)**</font> Although this feature appears as a count, we will treat this as a categorical. The effect of dependents on churn is likely not linear. For example, the difference in lifestyle/financial need between `0` and `1` dependent might be similar to the difference between `4` and `5`, or entirely different. Categorical encoding allows a model (like `RandomForestClassifier()` or `XGBoostClassifier()`) to learn specific, non-linear impacts for each specific count (e.g. `0`, `1`, `2`, `3`, `4`, `5`). So, we will combine `4` and `5` into a new category `4+`, which causes all sub-categories to occur with >=9% frequency in the combined train+validation data, and eliminates one which has a frequency of ~4.2%
4. Across all categories, the churn rate has a modest increase from ~14.7% (`0` dependents) to ~ 17.7% (`3` dependents). So, this too looks like a weak predictor of churn.

#### Education Level of Customer

In [ ]:
chart = vzu.plot_altair_bar_chart(
    get_grouped_churn_rate(df, "education_level"),
    xvar="frac_churned:Q",
    xvar2="total:Q",
    yvar="education_level:N",
    xtitle="Fraction of Churned Customers (%)",
    xtitle2="Number of Customers",
    ytitle=None,
    y_sort="-x",
    tooltip=[
        "education_level",
        "Churned",
        "Did not Churn",
        alt.Tooltip(
            "frac_churned:Q", title="Fraction Churned (%)", format=",.2f"
        ),
    ],
    tooltip2=[
        "education_level",
        alt.Tooltip("total:Q", title="Number of Customers", format=","),
        alt.Tooltip(
            "frac_total:Q", title="Fraction of Customers (%)", format=",.2f"
        ),
    ],
    ptitle=alt.TitleParams(
        text="Churn Rate is Similar Regardless of Customers' Education Level",
        anchor="start",
        align="left",
        dx=100,
        fontSize=18,
    ),
    x_scale="linear",
    fig_size=dict(width=275, height=125),
    save_params=dict(fpath=figures_dir / "fig_26_eda_education_bar.html"),
)
chart

**Observations**

1. The different levels of education level shows a mostly uniform churn rate. Most credit card customers are graduates.
2. <font color='darkred'>**(Re-Grouping Rare Categories)**</font> We will combine the two lowest frequency sub-categories *Post-Graduate* and *Doctorate* into a category called *Post-Graduate*, as both levels of education are similar to each other and are above all other sub-categories. This causes all sub-categories to occur with >10% frequency in the combined train+validation data, and eliminates two which each have a frequency of <~5%.
3. Excluding the two highest categories by churn rate, which are also the two rarest caregories in this feature, the churn rate is nearly unchanged increasing from ~15.1% (*College*) to ~16.3% (*Unknown*). This too looks like a poor predictor of credit card churn.

#### Income Category of Customer

In [ ]:
chart = vzu.plot_altair_bar_chart(
    get_grouped_churn_rate(df, "income_category"),
    xvar="frac_churned:Q",
    xvar2="total:Q",
    yvar="income_category:N",
    xtitle="Fraction of Churned Customers (%)",
    xtitle2="Number of Customers",
    ytitle=None,
    y_sort="-x",
    tooltip=[
        "income_category",
        "Churned",
        "Did not Churn",
        alt.Tooltip(
            "frac_churned:Q", title="Fraction Churned (%)", format=",.2f"
        ),
    ],
    tooltip2=[
        "income_category",
        alt.Tooltip("total:Q", title="Number of Customers", format=","),
        alt.Tooltip(
            "frac_total:Q", title="Fraction of Customers (%)", format=",.2f"
        ),
    ],
    ptitle=alt.TitleParams(
        text=(
            "Churn Rates for Two Highest Education Levels Are Similar That "
            "are Known"
        ),
        anchor="start",
        align="left",
        dx=105,
        fontSize=18,
    ),
    x_scale="linear",
    fig_size=dict(width=275, height=125),
    save_params=dict(fpath=figures_dir / "fig_27_eda_income_bar.html"),
)
chart

**Observations**

1. The *Less than $40K* group is the largest segment and it contributes the highest absolute number of churners.
2. <font color='darkred'>**(Re-Grouping Rare Categories)**</font> We will combine *80K-120K* and *120K +* since these are generally high-income customers. Similar to `education_level`, this causes all sub-categories to occur with >10% frequency in the combined train+validation data, and eliminates one which has a frequency of <~7%.
3. As with other non-numerical features, churn behavior appears relatively consistent across different income brackets. This also looks like a poor predictor of customer churn in the sample credit card customer data.

#### Summary

As we can see, there are some categorical and ordinal features with rare categories (categories that do not occur frequently in the dataset). They result in unnecessary high-dimensional, sparse transformed features that lead to models overfitting to noise instead of learning meaningful patterns. So, we recommend to either drop or group these rare categories to improve the usefulness of the feature as described per feature above.

### Handling Sensitive Features

Below are the sensitive features in the data

1. `age`
2. `gender`
3. `marital_status`
4. `dependent_count`
4. `education_level`
5. `income_category`

According to [Kamiran and Calders](https://link.springer.com/article/10.1007/s10115-011-0463-8), sensitive featyues can be used as predictive features. However, they must be handled through specific preprocessing techniques to avoid illegal and unethical discrimination of customers during a targeted marketing campaign (as is the overall use-case for this project) based on theses sensitive.

One recommended approach to ensure the predictive model does not unfairly discriminate based on age is to remove the sensitive feature and other features that correlate most with it, reducing the model's reliance on potentially discriminatory information. This approach will be used for `customer_age`, so it will be excluded from model development. Since `age` is not strongly correlated to other features as shown above, no other features need to be dropped in order to follow this approach.

`gender` is a *balanced* categorical feature that has two sub-categories that occur with almost the same frequency. So, we don't need to group any sub-categories and we will keep this feature unchanged.

Based on observations above, categorical and ordinal features do not appear to be predictors of churn on their own. We will run ML experiments to determine if the other sensitive features (which are either categorical or ordinal) can be dropped without negatively impacting model performance.

### Feature Lists by Type

Based on the EDA from above, features using in ML development should exclude the following

1. correlated numerical features
2. single-valued feature (`card_category`)
3. identifier features (`clientnum`)
4. sensitive feature (`customer_age`)

With this in mind, the ordinal, categorical and numerical features to be used in validation and evaluation are shown below

In [ ]:
ordinal_features = [
    "income_category",
    "education_level",
]

categorical_features = [
    "gender",
    "marital_status",
    "dependent_count",
]

# keep
# - 'credit_limit' and exclude "avg_open_to_buy" which is correlated
# - 'total_revolv_bal' and exclude "avg_utilization_ratio" which is correlated
numeric_features_1 = [
    "months_on_book",
    "num_products",
    "months_inactive_12_mon",
    "contacts_count_12_mon",
    "total_amt_chng_q4_q1",
    "total_trans_amt",
    "total_trans_ct",
    "total_ct_chng_q4_q1",
    "credit_limit",
    "total_revolv_bal",
]

# keep
# - 'avg_open_to_buy' and exclude "credit_limit" which is correlated
# - 'avg_utilization_ratio' and exclude "total_revolv_bal" which is correlated
numeric_features_2 = [
    "months_on_book",
    "num_products",
    "months_inactive_12_mon",
    "contacts_count_12_mon",
    "total_amt_chng_q4_q1",
    "total_trans_amt",
    "total_trans_ct",
    "total_ct_chng_q4_q1",
    "avg_open_to_buy",
    "avg_utilization_ratio",
]

Below is a summary of the features after transformation

In [ ]:
_ = show_df(df[ordinal_features + categorical_features + numeric_features_1])

## Conclusion

We identified that behavioral metrics (numerical features like usage frequency, transaction amounts, and utilization) are stronger predictors than static demographics (categorical or ordinal features such as age, education, etc.). Features like `total_trans_ct`, `total_revolv_bal`, and `avg_utilization_ratio` show a distinct separation between classes (churn versus no churn). This makes them high-priority candidates for ML model development.

The bivariate analysis revealed that churners aren't just "low users"; they often show a sharp **velocity decline**. The strong clustering in the `total_ct_chng_q4_q1` vs. `total_amt_chng_q4_q1` scatter plots suggests that creating additional 'momentum' features—such as ratios of activity over shorter time windows—could significantly improve model performance.

Based on these findings, our hypothesis is that numerical features are sufficient on their own to develop a ML model that can predict customer churn. this hypothesis should be verified during ML model validation (next notebook).

Our proposed pipeline involves using tree-based models (which are robust to the skewness observed in numerical feature distributions), cost-sensitive learning (with `class_weight='balanced'`) and `scikit-learn`'s `TunedThresholdClassifierCV()` to optimize for recall or f2-score (which prioritizes recall).